# A mutable companion table — a feature store you federate into queries

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/book/12-feature-store/feature-store.ipynb)

Built from [`cookbook/book/chapters/12-feature-store/feature-store.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/12-feature-store/feature-store.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures, from the release's
# tag on GitHub. The chapter runs
# at `small` scale, over the committed fixtures, in minutes. SCALE = "full" runs
# it over the published data and real encoders instead: meant for a GPU, and the
# chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook @ git+https://github.com/f-inverse/jammi-ai@py-v0.49.1#subdirectory=cookbook/book"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

In [ ]:
import jammi_cookbook

**Recipe:** `create_mutable_table` · `list_mutable_tables` · `sql`
(`INSERT` / `REPLACE INTO` / `UPDATE` / `DELETE` against `mutable.public.<name>`)
· the federating `JOIN` · **Theory:** the append-only result log vs mutable
companion state (Kleppmann 2017), the feature store as the substrate that
serves features into queries (Orr et al. 2021) · **Rail:** measurement (the
feature table stays equal to the feature recomputed from its source, through
every change).

The engine's result tables are **append-only** — a query writes a new immutable
Parquet result; nothing is edited in place. That is the right default for a
result log (Kleppmann 2017). A practitioner also needs the other half: a
piece of *mutable companion state* keyed by an entity id — a feature column —
that is kept current as its source changes and **federated into queries** by
name. That state is what a feature store is (Orr et al. 2021), and the
engine's primitive for it is the **mutable companion table**: provisioned with
a schema and a primary key, changed with ordinary SQL DML, joined into any
query.

The feature here is real and graph-derived: each paper's **citation in-degree**
— how many citations point *to* it. The chapter keeps that feature current
through the two things that happen to a citation graph: new citations arrive,
and a paper is withdrawn. After each change it checks the one property a feature
store must hold — **the stored feature equals the feature recomputed from the
source** — and measures the federated aggregate.

In [ ]:
import tempfile

import jammi
import pyarrow as pa
from jammi_cookbook import contracts, datasets, scale

SCALE = scale.current()
db = jammi.connect(f"file://{tempfile.mkdtemp()}")
arxiv = datasets.arxiv(db, SCALE)
PAPERS = f"{arxiv.papers}.public.{arxiv.papers}"
CITES = f"{arxiv.cites}.public.{arxiv.cites}"
FEATURES = "mutable.public.paper_features"

## Provision the companion table

A mutable table is provisioned with a `pyarrow` schema and a primary key, and
addressed in SQL as `mutable.public.<name>`. `list_mutable_tables()` is the
control-plane read of what is registered.

In [ ]:
db.create_mutable_table(
    "paper_features",
    schema=pa.schema([("paper_id", pa.string()), ("in_degree", pa.int64())]),
    primary_key=["paper_id"],
)
for t in db.list_mutable_tables():
    print(f"{t['id']}  primary_key={t['primary_key']}")

## Populate from the graph as it stood

The feature is derived in SQL from the citation source. To have something to
keep current, we first build it from the citations made **before the test era**
— those whose citing paper was published before 2019 — and let the newer
citations arrive in the next step.

In [ ]:
TEST_ERA = 2019


def in_degree(where: str) -> str:
    """The in-degree feature over the citations `where` selects."""
    return (
        f"SELECT c.dst AS paper_id, COUNT(*) AS in_degree FROM {CITES} c "
        f"JOIN {PAPERS} src ON c.src = src.paper_id WHERE {where} GROUP BY c.dst"
    )


def count(statement: str) -> int:
    return db.sql(statement).column("count")[0].as_py()


populated = count(f"INSERT INTO {FEATURES} {in_degree(f'src.year < {TEST_ERA}')}")
print(f"populated {populated} papers' in-degree from the pre-{TEST_ERA} citations")

## The one property: the store equals its source

A feature store is only as good as its agreement with the data it summarizes.
This check recomputes the feature from the citation source and compares it, row
for row, with what the companion table holds — both sides in SQL, federated in
one query.

In [ ]:
def drift(live: str) -> int:
    """Rows where the stored feature and the feature recomputed over `live` disagree."""
    return db.sql(
        f"SELECT COUNT(*) AS n FROM {FEATURES} f "
        f"FULL OUTER JOIN ({in_degree(live)}) s ON f.paper_id = s.paper_id "
        f"WHERE COALESCE(f.in_degree, 0) <> COALESCE(s.in_degree, 0)"
    ).column("n")[0].as_py()


print(f"drift against the pre-{TEST_ERA} graph: {drift(f'src.year < {TEST_ERA}')}")

In [ ]:
assert drift(f"src.year < {TEST_ERA}") == 0

## New citations arrive — `REPLACE INTO`

The test-era papers are published, and their citations arrive. Only the papers
they cite need a new value; `REPLACE INTO` upserts by primary key — an existing
paper's row is replaced with its new in-degree, a newly-cited paper's row is
inserted.

In [ ]:
touched = (
    f"c.dst IN (SELECT c2.dst FROM {CITES} c2 JOIN {PAPERS} p2 ON c2.src = p2.paper_id "
    f"WHERE p2.year >= {TEST_ERA})"
)
upserted = count(f"REPLACE INTO {FEATURES} {in_degree(touched)}")
print(f"upserted {upserted} papers; drift against the full graph: {drift('TRUE')}")

In [ ]:
assert drift("TRUE") == 0

## A paper is withdrawn — `UPDATE` and `DELETE`

When a paper is withdrawn, two things change: the papers it cited each lose one
citation (`UPDATE`), and its own feature row goes (`DELETE`). We withdraw the
test-era paper that cites the most others.

An `UPDATE` or `DELETE` chooses its rows by a predicate over the table's own
columns; the engine refuses one that reaches into another relation (a subquery
or a join), because it could not honour it. So we first read the keys the
withdrawal touches, then name them.

In [ ]:
withdrawn = db.sql(
    f"SELECT c.src FROM {CITES} c JOIN {PAPERS} p ON c.src = p.paper_id "
    f"WHERE p.year >= {TEST_ERA} GROUP BY c.src ORDER BY COUNT(*) DESC, c.src LIMIT 1"
).column("src")[0].as_py()
cited = db.sql(f"SELECT DISTINCT dst FROM {CITES} WHERE src = '{withdrawn}'").column("dst")
keys = ", ".join(f"'{k}'" for k in cited.to_pylist())

decremented = count(
    f"UPDATE {FEATURES} SET in_degree = in_degree - 1 WHERE paper_id IN ({keys})"
)
deleted = count(f"DELETE FROM {FEATURES} WHERE paper_id = '{withdrawn}'")
survivors = f"c.src <> '{withdrawn}' AND c.dst <> '{withdrawn}'"
print(f"withdrew {withdrawn}: {decremented} papers lost a citation, {deleted} row deleted")
print(f"drift against the surviving graph: {drift(survivors)}")

In [ ]:
assert drift(survivors) == 0
contracts.assert_close("feature_store.populated_rows", populated)
contracts.assert_close("feature_store.upserted_rows", upserted)
contracts.assert_close("feature_store.withdrawn_citations", decremented)

## Federate the feature into a query

The payoff: the companion table JOINs into a query over the registered papers
source like any other table. The total citation in-degree per subject:

In [ ]:
agg = db.sql(
    f"SELECT p.subject AS subject, SUM(f.in_degree) AS total FROM {PAPERS} p "
    f"JOIN {FEATURES} f ON p.paper_id = f.paper_id "
    "GROUP BY p.subject ORDER BY total DESC, subject"
).to_pylist()
total = sum(r["total"] for r in agg)
edges = db.sql(f"SELECT COUNT(*) AS n FROM {CITES} c WHERE {survivors}").column("n")[0].as_py()
for r in agg[:6]:
    print(f"{r['subject']:<16}{r['total']:>8}")
print(f"\n{len(agg)} subjects · total in-degree {total} · surviving citations {edges}")

In [ ]:
assert total == edges  # every surviving citation is counted exactly once
contracts.assert_close("feature_store.total_in_degree", total)
contracts.assert_close("feature_store.top_subject_total", agg[0]["total"])

The subject totals sum back to the surviving citation count: each citation
contributes one unit of in-degree to the paper it cites, and every change —
the arrival, the withdrawal — kept that true.

In [ ]:
db.close()

## Bridge note

> **A result log and a feature store are two sides of one storage primitive.**
> The engine writes query results as an **append-only** log of immutable
> Parquet — the right default for derived results (Kleppmann 2017). A
> **mutable companion table** is the complement: registered state keyed by an
> entity id, kept current with `REPLACE INTO` / `UPDATE` / `DELETE`, and
> **federated** into queries by name — the role a feature store plays
> (Orr et al. 2021). The measured lesson is the feature store's one
> obligation: after every change, the stored feature equals the feature
> recomputed from its source.

## References

- Kleppmann, Martin (2017) *Designing Data-Intensive Applications: The Big Ideas Behind Reliable, Scalable, and Maintainable Systems* O'Reilly Media.
- Orr, Laurel, Sanyal, Atindriyo, Ling, Xiao, Goel, Karan, Leszczynski, Megan (2021) *Managing ML Pipelines: Feature Stores and the Coming Wave of Embedding Ecosystems* Proceedings of the VLDB Endowment DOI 10.14778/3476311.3476402.